In [1]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, f1_score
from sklearn.exceptions import ConvergenceWarning
import warnings

warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [2]:
# Constants
TRAIN_SIZE = 0.2
RANDOM_STATE = 1082

base_dir = "../../../Datasets/csv/UAVATTACK/"
real_data_path = "../../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv"
RESULTS_CSV = "../../../UAVATTACK_Results.csv"
models_list = ["ctgan", "gc", "tvae"]
sample_sizes = [10_000, 100_000, 1_000_000]

# Also include any other files you might want to test
other = [
    "syntheticSpoofing_cgan_data.csv",
    "syntheticSpoofing_gan_data.csv"
]

In [3]:
# Load synthetic datasets
synthetic_dfs = {}
# Try all combinations of model/sample size
for model in models_list:
    for n in sample_sizes:
        file_name = f"spoofing_{model}_{n}.csv"
        file_path = os.path.join(base_dir, file_name)
        try:
            synthetic_dfs[file_name] = pd.read_csv(file_path)
            print(f"Loaded {file_name}")
        except Exception as e:
            print(f"Could not load {file_name}: {e}")

# Add other synthetic dataset
for file_name in other:
    file_path = os.path.join(base_dir, file_name)
    try:
        synthetic_dfs[file_name] = pd.read_csv(file_path)
        print(f"Loaded {file_name}")
    except Exception as e:
        print(f"Could not load {file_name}: {e}")

# Load real data
try:
    real_df = pd.read_csv(real_data_path)
    real_df.columns = real_df.columns.str.replace('[\\[\\]<>]', '_', regex=True)
    print("Loaded real data")
except Exception as e:
    print(f"Could not load real data: {e}")
    real_df = None

Loaded spoofing_ctgan_10000.csv
Loaded spoofing_ctgan_100000.csv
Loaded spoofing_ctgan_1000000.csv
Loaded spoofing_gc_10000.csv
Loaded spoofing_gc_100000.csv
Loaded spoofing_gc_1000000.csv
Loaded spoofing_tvae_10000.csv
Loaded spoofing_tvae_100000.csv
Loaded spoofing_tvae_1000000.csv
Loaded syntheticSpoofing_cgan_data.csv
Loaded syntheticSpoofing_gan_data.csv
Loaded real data


In [4]:
# Preprocess real data
if real_df is not None:
    if 'label' not in real_df.columns:
        raise ValueError("Real data does not have 'label' column.")
    X_real = real_df.drop(columns=[col for col in ['label', 'timestamp'] if col in real_df.columns])
    y_real = real_df['label']
    if y_real.dtype == object or y_real.dtype.name == 'category':
        le_real = LabelEncoder()
        y_real = le_real.fit_transform(y_real)
else:
    raise RuntimeError("Real data could not be loaded.")

In [5]:
# Models to evaluate
models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        n_jobs=-1,
        random_state=RANDOM_STATE
    )
}

In [6]:
results = []
for synth_name, synth_df in synthetic_dfs.items():
    print(f"Training on synthetic: {synth_name}, testing on real data")

    if 'label' not in synth_df.columns:
        print(f"Skipping {synth_name}: 'label' column not found.")
        continue

    # Drop label and timestamp from synthetic, match columns to real data
    X_synth = synth_df.drop(columns=[col for col in ['label', 'timestamp'] if col in synth_df.columns])
    y_synth = synth_df['label']
    if y_synth.dtype == object or y_synth.dtype.name == 'category':
        le_synth = LabelEncoder()
        y_synth = le_synth.fit_transform(y_synth)

    X_synth_train, _, y_synth_train, _ = train_test_split(
        X_synth, y_synth, train_size=TRAIN_SIZE, random_state=RANDOM_STATE, stratify=y_synth
    )

    # Scale based on synthetic training split, apply to both train and real test
    scaler = StandardScaler()
    X_synth_train_scaled = scaler.fit_transform(X_synth_train)
    X_real_scaled = scaler.transform(X_real)

    for model_name, model in models.items():
        print(f"\n{model_name}")
        print("Training on synthetic")
        model.fit(X_synth_train_scaled, y_synth_train)
        print("Evaluating on real data...")
        y_pred = model.predict(X_real_scaled)
        acc = accuracy_score(y_real, y_pred)
        f1 = f1_score(y_real, y_pred, average='weighted', zero_division=0)
        print(f"Accuracy: {acc:.4f}")
        print(f"F1 Score: {f1:.4f}")
        print("\nClassification Report:")
        print(classification_report(y_real, y_pred, zero_division=0))
        print("Confusion Matrix:")
        print(confusion_matrix(y_real, y_pred))

        # Save results
        results.append({
            "model": model_name,
            "dataset": synth_name,
            "f1score": f1,
            "accuracy": acc
        })

Training on synthetic: spoofing_ctgan_10000.csv, testing on real data

Random Forest
Training on synthetic
Evaluating on real data...
Accuracy: 0.9660
F1 Score: 0.9640

Classification Report:
              precision    recall  f1-score   support

           0       0.96      1.00      0.98      3124
           1       1.00      0.75      0.86       498

    accuracy                           0.97      3622
   macro avg       0.98      0.88      0.92      3622
weighted avg       0.97      0.97      0.96      3622

Confusion Matrix:
[[3124    0]
 [ 123  375]]
Training on synthetic: spoofing_ctgan_100000.csv, testing on real data

Random Forest
Training on synthetic
Evaluating on real data...
Accuracy: 0.9649
F1 Score: 0.9627

Classification Report:
              precision    recall  f1-score   support

           0       0.96      1.00      0.98      3124
           1       1.00      0.74      0.85       498

    accuracy                           0.96      3622
   macro avg       0.98  

In [7]:
# Save to CSV
results_df = pd.DataFrame(results)
file_exists = os.path.isfile(RESULTS_CSV)
results_df.to_csv(
    RESULTS_CSV,
    mode='a',
    header=not file_exists,
    index=False
)
print(f"Results appended to {RESULTS_CSV}")

Results appended to ../../../UAVATTACK_Results.csv
